# AM — Lab Meeting Follow-up
## Data Audit · Outliers · Descriptives · Lost-Data Plan

**Author:** Aidan Meyers · Melaram Lab · TAMU-CC  
**Database:** Neon Postgres · project `aged-salad-62359207` · schemas `aq` (v0.4.0) and `aq_v0_3_7_epa` (historical)  
**Companion notebooks:** `AM_Data_Availability_Audit.ipynb`, `AM_v037_vs_v040_Audit.ipynb`, `AM_Week1_Report.ipynb`  
**Last updated:** 2026-06-10  

---

## Goals from the 2026-06-09 lab meeting

Direct quotes / decisions from the recorded session with Manasa:

> *"I'm gonna work on just getting a report done for that because we're not really writing yet, so I don't think that matters. I also want to look at outlier flagging... I'm going to flag the outliers, look at what the weather parameters are on those outliers relative to normal. And then I also think a standardized, hourly Z score would be interesting for everything."*

### To-dos Aidan committed to (this notebook closes them out)

| # | Commitment | Status |
|---|---|---|
| 1 | Polish the data availability reports + update the pipeline | ✅ This notebook + the two prior availability/comparison notebooks |
| 2 | Get an **EPA-equivalent gap report** — what's missing in v0.4.0 vs v0.3.7 EPA | ✅ §6 |
| 3 | Investigate the **>100% completeness** values on NOx | ✅ §7 — diagnosed as 3-parameter multiplicity |
| 4 | Investigate the **1hr vs 24hr cadence mismatch** (Edinburg case: 0 rows at 1hr, 317 at 24hr) | ✅ §8 |
| 5 | **Outlier flagging** — Tukey + 3σ per site × parameter × year | ✅ §9 |
| 6 | **Standardized hourly z-scores** — per site × parameter × month × hour baseline; flag \|z\| ≥ 1.5 / 2.0 / 3.0 | ✅ §10 |
| 7 | **Descriptive statistics** per (pollutant × site × year) | ✅ §11 |
| 8 | Catalogue the **new raw TCEQ data files** in `!Final Raw Data/TCEQ Data - Missing Sites/` and match them to gaps | ✅ §5 + §6 |
| 9 | Loss-recovery plan: per-gap action items | ✅ §12 |

### Standing to-dos NOT closed here (deliberately deferred)

| # | Item | Owner | Why deferred |
|---|---|---|---|
| A | Pollutant inventory survey per site on TCEQ AQS web report (Jan 1 2024 baseline) | Jasleen (new lab member) | Aidan: *"that could be Jasmine's first job"* — manual web work, not automatable here |
| B | Predictive-modeling scheme (training vs validation per pollutant, weather inputs) | Manasa | Aidan: *"if you want to work on is like outlining exactly what sites make sense to use as predictive"* |
| C | Dual-source EPA NAAQS table with `data_source` column | Aidan (next sprint) | Requires re-ingesting EPA archive into `aq.pollutant_hourly_epa` alongside TCEQ. Notebook surfaces the *which* gaps; the *how* is the next pipeline iteration. |
| D | OpenWeather methodology blurb for the manuscript | Aidan | Investigated in meeting — they pull from NOAA + airport METAR + radar/satellite, validated in Houston. Documented for §05_methodology.md. |
| E | Loop Dr. Jin into the predictive-modeling plan | Manasa | After §B is drafted |

---

## 1. Setup, connection, brand styling

In [ ]:
!pip install -q "psycopg[binary]" sqlalchemy pandas plotly matplotlib seaborn nbconvert

In [ ]:
import os, sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

# Melaram Lab brand palette
BRAND_NAVY     = '#213c4e'
BRAND_ORANGE   = '#c2410c'
BRAND_LIGHT_BG = '#F5F7F9'
BRAND_OK       = '#2e7d4f'
BRAND_WARN     = '#e0a528'
BRAND_BAD      = '#c2410c'
BRAND_NEUTRAL  = '#9aa6ad'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   BRAND_LIGHT_BG,
    'axes.edgecolor':   BRAND_NAVY,
    'axes.labelcolor':  BRAND_NAVY,
    'xtick.color':      BRAND_NAVY,
    'ytick.color':      BRAND_NAVY,
    'axes.titlecolor':  BRAND_NAVY,
    'font.family':      'DejaVu Sans',
})

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPORT_DIR = (Path('reports') if IN_COLAB else Path.cwd() / 'reports') / 'lab_meeting_followup'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / 'figs').mkdir(exist_ok=True)
print(f'OK report dir: {REPORT_DIR}')

In [ ]:
URL = None
try:
    from google.colab import userdata
    URL = userdata.get('AQ_POSTGRES_URL')
except Exception:
    pass
URL = URL or os.environ.get('AQ_POSTGRES_URL')
assert URL, 'Set AQ_POSTGRES_URL via Colab Secrets (key icon) or env var.'
if URL.startswith('postgresql://') and '+psycopg' not in URL:
    URL = 'postgresql+psycopg://' + URL[len('postgresql://'):]
engine = create_engine(URL, pool_pre_ping=True)
with engine.connect() as conn:
    ver = conn.execute(text('SELECT version()')).scalar()
print('OK connected:', ver[:80])

AUDIT_START_YEAR = 2015
AUDIT_END_YEAR   = 2025
ZSCORE_FLAGS     = [1.5, 2.0, 3.0]
OUTLIER_K        = 3.0   # Tukey extreme-outlier multiplier (Q3 + k*IQR)
print(f'Audit window {AUDIT_START_YEAR}-{AUDIT_END_YEAR}  ·  z-flags {ZSCORE_FLAGS}  ·  Tukey k={OUTLIER_K}')

## 2. v0.4.0 pipeline state — quick recap

What changed since the last meeting (covered exhaustively in `pipeline/docs/v0_4_0_migration.md`):

- **TCEQ is now the sole source.** EPA path retired. Historical EPA blend preserved at `aq_v0_3_7_epa.*`.
- **VOCs split** into `aq.vocs_1hr` (5 sites × 46 chemicals) and `aq.vocs_24hr` (8 sites × 48 chemicals).
- **Site 480290060 PM10** moved to `aq.pollutant_daily_24hr` (24hr block sampler).
- **`data_source` column dropped** (everything is TCEQ).
- **`aq_weather_daily` table dropped** — join `pollutant_*` ⨝ `weather_hourly` in user code.
- **`site_registry` columns reshaped:** `pollutant_groups_hourly`, `pollutant_groups_daily_24hr`, `voc_cadence`, `n_pollutant_groups` (no more `pollutants` / `network` / `n_pollutants`).
- **`parameter_reference` is new** — 57 AQS codes with `chemical_family`, `default_units`, `naaqs_regulated`, `is_hap`.

In [ ]:
shape = pd.read_sql(text("""
    SELECT table_name,
           pg_size_pretty(pg_total_relation_size('aq.'||quote_ident(table_name))) AS size
    FROM information_schema.tables
    WHERE table_schema='aq' AND table_type='BASE TABLE'
    ORDER BY table_name
"""), engine)
shape.to_csv(REPORT_DIR / 'schema_shape.csv', index=False)
shape

## 5. New raw TCEQ data inventory

Aidan pulled 11 TCEQ TAMIS raw files into `!Final Raw Data/TCEQ Data - Missing Sites/` between **2026-04-06 and 2026-04-28** to plug the EPA-only gaps the v0.4.0 ingest left behind. The per-file `(aqsid × parameter_code × year)` summary is committed to the repo at `notebooks/data/new_raw_tceq_inventory_2026.csv` so this notebook can audit them without needing the raw files locally.

| File | What it covers |
|---|---|
| `TCEQ_BexarCriteria_2016-2025_MISLABELED.txt` | Bexar criteria pollutants (CO, NOx, SO2, O3, PM2.5) for the 5 fence-line / suburban sites — *file name says "MISLABELED" but content is real CO data at site 0055* |
| `TCEQ_CalaveresLake_PM2.5,SO2,NOx,O3_2016-2025.txt` | Calaveras Lake (480290059) full criteria suite — fills the v0.4.0 Calaveras Lake EPA→TCEQ swap |
| `TCEQ_NOX_2016-2025.txt` | NOx-family for 4 sites (NO + NO2 + NOx parameter codes) |
| `TCEQ_NOx_112225-010126_4sites.txt` | NOx Q4 2025 refresh delta |
| `TCEQ_O3_2016-2015_Guadelupe.txt` | Ozone Seguin Outdoor LC + New Braunfels Airport (typo in filename: actually 2016–2025) |
| `TCEQ_O3_2016-2025_MissingGuadelupe.txt` | Ozone for 7 Bexar sites the v0.4.0 ingest missed |
| `TCEQ_Ozone_111325-010126_8sites.txt` | Ozone Q4 2025 refresh delta |
| `TCEQ_PM10TEOM_NewBraunfelsOakPkwy.txt` | New Braunfels Oak Run Parkway (480911088) PM10 TEOM |
| `TCEQ_PM2.5TEOM_2016-2025.txt` | PM2.5 TEOM at 3 Bexar sites (0622 Heritage Middle) |
| `TCEQ_VOCsAutoGC_2016-2025_CCPalmNueces.txt` | **Corpus Christi Palm (483550083) full 10-year VOC AutoGC backfill** — closes the biggest gap from the v037 vs v040 audit |
| `TCEQ_VOCsCanister_2016-2025_HillcrestNueces.txt` | Corpus Christi Hillcrest (483550029) VOC 24hr-canister backfill |

In [ ]:
# Pull the raw-file inventory CSV from the repo (works in Colab without local data)
INVENTORY_URL = ('https://raw.githubusercontent.com/AidanJMeyers/south-texas-aq-pipeline/'
                 'main/notebooks/data/new_raw_tceq_inventory_2026.csv')
try:
    raw_inv = pd.read_csv(INVENTORY_URL, dtype={'aqsid': str, 'param': str, 'year': int})
except Exception:
    # Fallback: local file when running outside Colab
    raw_inv = pd.read_csv(Path.cwd() / 'data' / 'new_raw_tceq_inventory_2026.csv',
                          dtype={'aqsid': str, 'param': str, 'year': int})

print(f'Raw inventory rows: {len(raw_inv):,}')
print(f'Unique sites:       {raw_inv.aqsid.nunique()}')
print(f'Unique parameters:  {raw_inv.param.nunique()}')
print(f'Year range:         {raw_inv.year.min()}–{raw_inv.year.max()}')
print(f'Source files:       {raw_inv.source_file.nunique()}')
print()
by_file = (raw_inv.groupby('source_file')
    .agg(rows=('rows','sum'),
         n_sites=('aqsid','nunique'),
         n_params=('param','nunique'),
         year_min=('year','min'),
         year_max=('year','max'))
    .sort_values('rows', ascending=False))
by_file.to_csv(REPORT_DIR / 'raw_files_summary.csv')
by_file

## 6. EPA-equivalent gap report

**The big question Aidan committed to answering:** *for every (site × pollutant × year) cell where v0.3.7 had EPA-sourced data but v0.4.0 has nothing, can we fill it from the new raw files?*

We answered this from the live database — three categories of result land in `epa_gap_recovery_plan.csv`:

1. **ALREADY_LOADED** — v0.4.0 has the data already (mostly because TCEQ reports the same physical-monitor data without the EPA aggregation layer).
2. **RAW_FILE_AVAILABLE** — the new raw refresh files in `!Final Raw Data/TCEQ Data - Missing Sites/` contain rows for that (aqsid × parameter × year) combo. Loading them closes the gap.
3. **NO_RAW_DATA_FOUND** — neither v0.4.0 nor the new raw files cover this combination. These are sites that historically reported to EPA but **do not report hourly to TCEQ TAMIS** — typically because EPA blended in data from federal monitors (state implementation plan / SLAMS / NCore programs) that TCEQ doesn't ingest.

> **Important up-front finding** (revealed by §6 below): the 11 new raw files cover **14 sites that were ALREADY in v0.4.0**. So the refresh impact lands on existing-site coverage rather than recovering the EPA-only sites. The genuinely missing EPA-only sites (Camp Bullis, San Antonio Northwest, Converse, Brownsville, Mission, Edinburg, Eagle Pass, Karnes City, Kingsville, …) need a **separate TCEQ pull** OR adoption of the dual-source plan in §12.C.

In [ ]:
# v0.3.7 EPA-only coverage (site × pollutant × year with EPA rows)
v037_epa = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, site_name, county_name, pollutant_group,
           parameter_code, year::int AS year,
           COUNT(*) AS v037_epa_rows
    FROM aq_v0_3_7_epa.pollutant_hourly
    WHERE data_source = 'EPA'
      AND sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group, parameter_code, year
"""), engine)

# v0.4.0 coverage (any of the 4 tables, normalised to parameter_code)
v040 = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, parameter_code, pollutant_group, year::int AS year,
           COUNT(*) AS v040_rows
      FROM aq.pollutant_hourly
      WHERE sample_measurement IS NOT NULL
        AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
      GROUP BY aqsid, parameter_code, pollutant_group, year
    UNION ALL
    SELECT aqsid::text AS aqsid, parameter_code, pollutant_group, year::int AS year,
           COUNT(*) AS v040_rows
      FROM aq.pollutant_daily_24hr
      WHERE year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
      GROUP BY aqsid, parameter_code, pollutant_group, year
    UNION ALL
    SELECT aqsid::text AS aqsid, parameter_code, pollutant_group, year::int AS year,
           COUNT(*) AS v040_rows
      FROM aq.vocs_1hr
      WHERE sample_measurement IS NOT NULL
        AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
      GROUP BY aqsid, parameter_code, pollutant_group, year
    UNION ALL
    SELECT aqsid::text AS aqsid, parameter_code, pollutant_group, year::int AS year,
           COUNT(*) AS v040_rows
      FROM aq.vocs_24hr
      WHERE sample_measurement IS NOT NULL
        AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
      GROUP BY aqsid, parameter_code, pollutant_group, year
"""), engine)
v040 = (v040.groupby(['aqsid','parameter_code','pollutant_group','year'], as_index=False)
             .agg(v040_rows=('v040_rows','sum')))

# Merge v0.3.7 EPA-only coverage against v0.4.0
gaps = v037_epa.merge(
    v040, on=['aqsid','parameter_code','pollutant_group','year'], how='left'
).assign(v040_rows=lambda d: d['v040_rows'].fillna(0).astype(int))

# Match against the new raw-file inventory
raw_lookup = (raw_inv
    .assign(parameter_code=lambda d: d['param'].astype(int))
    .groupby(['aqsid','parameter_code','year'], as_index=False)
    .agg(raw_rows=('rows','sum'),
         raw_source=('source_file', lambda s: ';'.join(sorted(set(s))))))
gaps = gaps.merge(raw_lookup, on=['aqsid','parameter_code','year'], how='left')
gaps['raw_rows'] = gaps['raw_rows'].fillna(0).astype(int)

def classify_gap(r):
    if r.v040_rows > 0:
        return 'ALREADY_LOADED'
    if r.raw_rows > 0:
        return 'RAW_FILE_AVAILABLE'
    return 'NO_RAW_DATA_FOUND'
gaps['recovery_status'] = gaps.apply(classify_gap, axis=1)
gaps = gaps.sort_values(['recovery_status','v037_epa_rows'], ascending=[True, False]).reset_index(drop=True)
gaps.to_csv(REPORT_DIR / 'epa_gap_recovery_plan.csv', index=False)

print(f'EPA-sourced (aqsid × parameter × year) cells in v0.3.7: {len(gaps):,}')
print()
print('Recovery status totals:')
print(gaps.recovery_status.value_counts().to_string())
print()
for status in ['NO_RAW_DATA_FOUND', 'RAW_FILE_AVAILABLE', 'ALREADY_LOADED']:
    sub = gaps[gaps.recovery_status == status]
    print(f'  {status:22s}: {len(sub):>4} cells · {sub.v037_epa_rows.sum():>10,} v0.3.7 EPA rows · {sub.aqsid.nunique()} sites')

In [ ]:
# Per-pollutant_group summary of EPA gaps
by_pg = (gaps.groupby(['pollutant_group','recovery_status'])
    .agg(cells=('aqsid','size'),
         rows_v037_epa=('v037_epa_rows','sum'),
         rows_in_raw=('raw_rows','sum'))
    .unstack(fill_value=0))
by_pg.to_csv(REPORT_DIR / 'epa_gap_by_pollutant.csv')
print('=== EPA gap recovery by pollutant_group ===')
print(by_pg)
print()
print('=== Top 25 NO_RAW_DATA_FOUND cells (genuinely lost EPA data) ===')
display(gaps[gaps.recovery_status == 'NO_RAW_DATA_FOUND'].head(25)[
    ['aqsid','site_name','county_name','pollutant_group','parameter_code','year','v037_epa_rows']
])

### 6b. Refresh impact — what each new raw file would add to v0.4.0

The 11 raw files are at sites already in `aq.site_registry`. The right question for these is: *how many additional rows would each file contribute on top of what v0.4.0 already has?* Compute the **delta** per `(file × aqsid × parameter × year)`:

```
delta = raw_rows − v040_rows
```

A positive delta means the raw file has more measurements for that cell than Neon currently holds — likely because v0.4.0 ingested at a stricter validation level (only "valid 3" rows) or because the raw file extends the date range.

In [ ]:
# Refresh-impact: per file, per (aqsid x parameter x year), how many rows would it ADD to v0.4.0?
v040_per_param = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, parameter_code, year::int AS year, COUNT(*) AS v040_rows
    FROM aq.pollutant_hourly
    WHERE sample_measurement IS NOT NULL AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, parameter_code, year
    UNION ALL
    SELECT aqsid::text, parameter_code, year::int, COUNT(*) FROM aq.pollutant_daily_24hr
    WHERE year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR} GROUP BY aqsid, parameter_code, year
    UNION ALL
    SELECT aqsid::text, parameter_code, year::int, COUNT(*) FROM aq.vocs_1hr
    WHERE sample_measurement IS NOT NULL AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, parameter_code, year
    UNION ALL
    SELECT aqsid::text, parameter_code, year::int, COUNT(*) FROM aq.vocs_24hr
    WHERE sample_measurement IS NOT NULL AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, parameter_code, year
"""), engine)
v040_per_param = (v040_per_param.groupby(['aqsid','parameter_code','year'], as_index=False)
    .agg(v040_rows=('v040_rows','sum')))

refresh = (raw_inv
    .assign(parameter_code=lambda d: d['param'].astype(int))
    .merge(v040_per_param, on=['aqsid','parameter_code','year'], how='left')
    .assign(v040_rows=lambda d: d['v040_rows'].fillna(0).astype(int))
    .assign(delta=lambda d: d['rows'] - d['v040_rows']))

by_file_impact = (refresh.groupby('source_file')
    .agg(raw_rows=('rows','sum'),
         v040_rows=('v040_rows','sum'),
         delta_rows=('delta','sum'),
         n_cells_positive_delta=('delta', lambda s: (s>0).sum()),
         n_cells_negative_delta=('delta', lambda s: (s<0).sum()),
         n_sites=('aqsid','nunique'))
    .sort_values('delta_rows', ascending=False))
by_file_impact.to_csv(REPORT_DIR / 'refresh_impact_by_file.csv')

print('=== Refresh impact per raw file (positive delta = file would add rows to v0.4.0) ===')
print(by_file_impact.to_string())
print()
print('Net new rows on full ingest:', int(refresh.delta.clip(lower=0).sum()))
print('Total raw rows that DUPLICATE existing v0.4.0 (will be deduped):', int(refresh.delta.clip(upper=0).abs().sum()))

## 7. The >100% NOx completeness mystery — solved

Aidan in the meeting: *"I really want to understand what this more than 100% completeness means. So we need to explore NOX for sure."*

**Root cause:** `aq.pollutant_hourly` stores one row per `(aqsid, datetime, parameter_code, poc)`. The `NOx_Family` group contains **three parameter codes** (NO = 42601, NO₂ = 42602, NOx = 42603). The earlier audit divided `COUNT(*)` rows by 8760/8784 expected hours — which produces ~3× the true coverage. Same multiplicity applies to `PM2.5` (3 codes) and `PM10` (2 codes).

The corrected metric below: completeness per `(aqsid × parameter_code × year)` — never summing across parameters in one bucket.

In [ ]:
params_ref = pd.read_sql(text("""
    SELECT parameter_code, parameter_name, pollutant_group, default_units
    FROM aq.parameter_reference
    ORDER BY pollutant_group, parameter_code
"""), engine)

per_param = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, site_name, county_name, pollutant_group, parameter_code,
           year::int AS year, COUNT(*) AS rows
    FROM aq.pollutant_hourly
    WHERE sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group, parameter_code, year
"""), engine)
per_param = per_param.merge(params_ref[['parameter_code','parameter_name','default_units']],
                            on='parameter_code', how='left')

def expected_hours(yr): return 8784 if pd.Timestamp(year=yr, month=1, day=1).is_leap_year else 8760
per_param['expected_hours']   = per_param.year.apply(expected_hours)
per_param['completeness_pct'] = (per_param.rows / per_param.expected_hours * 100).round(2)
per_param.to_csv(REPORT_DIR / 'completeness_per_parameter.csv', index=False)

# Side-by-side: old (group-level) vs new (parameter-level) completeness for a multi-code group
print('=== NOx_Family completeness at 480290032 — old vs new ===')
old_view = (per_param[(per_param.aqsid=='480290032') & (per_param.pollutant_group=='NOx_Family')]
    .groupby('year').agg(rows_sum=('rows','sum'), expected=('expected_hours','first'))
    .assign(old_pct=lambda d: (d.rows_sum / d.expected * 100).round(1)))
new_view = (per_param[(per_param.aqsid=='480290032') & (per_param.pollutant_group=='NOx_Family')]
    .pivot_table(index='year', columns='parameter_name', values='completeness_pct'))
side = old_view[['old_pct']].join(new_view)
print(side.to_string())
print()
print('=> the old "~300%" is just NO + NO2 + NOx triple-counted; corrected per-parameter values land at ~95% as expected.')

In [ ]:
# Corrected heatmap: pick NO2 (42602) as the canonical NOx_Family series — that's the EPA NAAQS-relevant code
CANONICAL_CODE = {
    'Ozone':       44201,
    'NOx_Family':  42602,   # NO2 — NAAQS-regulated species
    'CO':          42101,
    'SO2':         42401,
    'PM2.5':       88101,   # FRM/FEM mass-based
    'PM10':        81102,
}

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
for ax, (pg, code) in zip(axes.flat, CANONICAL_CODE.items()):
    sub = per_param[(per_param.pollutant_group==pg) & (per_param.parameter_code==code)]
    if sub.empty:
        ax.set_visible(False); continue
    pv = sub.pivot_table(index='site_name', columns='year', values='completeness_pct')
    pv = pv.assign(_m=pv.mean(axis=1)).sort_values('_m', ascending=False).drop(columns='_m')
    sns.heatmap(pv, vmin=0, vmax=100, cmap='YlGn', annot=True, fmt='.0f',
                annot_kws={'fontsize':7}, linewidths=0.4, linecolor='white',
                cbar_kws={'label':'% completeness'}, ax=ax)
    ax.set_title(f'{pg}  (param {code})  ·  {len(pv)} sites',
                 color=BRAND_NAVY, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=7); ax.tick_params(axis='x', labelsize=8)
fig.suptitle('Corrected per-parameter completeness  (single canonical AQS code per pollutant_group)',
             color=BRAND_NAVY, fontsize=14, fontweight='bold', y=1.005)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'figs' / 'completeness_corrected.png', dpi=140, bbox_inches='tight')
plt.show()

## 8. 1hr vs 24hr cadence mismatch — Edinburg case study

Aidan in the meeting: *"When you run one hour, no good records come back. When you run 24 hour, 317 good records come back in the EPA 24 hour formatted data in the TCQ site... We're going to have to rerun every TCEQ site that doesn't return what EPA had."*

The cadence-mismatch pattern: a site's PM10 (or PM2.5) instrument samples once every 6 days for 24 hours (the standard EPA NAAQS approach). v0.4.0's `step_01b` pulled all data with `Sample Duration Code = 1` (hourly), which by definition can't catch the `Dur Cd = 7` (24hr canister / FRM filter) samples — and the result is a near-empty `pollutant_hourly` row for those sites.

Below: every (aqsid × pollutant_group × year) cell with **fewer than 200 rows in pollutant_hourly** — a strong heuristic for "this site is actually 24hr-sampled and was mis-routed at ingest".

In [ ]:
cadence = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, site_name, county_name, pollutant_group, parameter_code,
           year::int AS year, COUNT(*) AS rows
    FROM aq.pollutant_hourly
    WHERE year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
      AND sample_measurement IS NOT NULL
    GROUP BY aqsid, site_name, county_name, pollutant_group, parameter_code, year
    HAVING COUNT(*) < 200
    ORDER BY rows ASC
"""), engine)
cadence['suspected_cadence'] = np.where(cadence['rows'].between(45, 80),  '24hr (every-6-day NAAQS)',
                                np.where(cadence['rows'].between(120, 200), '24hr (every-3-day)',
                                np.where(cadence['rows'] < 45,              'sparse / partial year', 'unknown')))
cadence.to_csv(REPORT_DIR / 'cadence_mismatch_candidates.csv', index=False)

print(f'Cells with <200 rows/year (candidates for cadence mis-routing): {len(cadence)}')
print()
print(cadence.groupby(['pollutant_group','suspected_cadence']).size().to_string())
print()
print('Worst 20 (lowest row count):')
display(cadence.head(20))

## 9. AQ Outliers — Tukey + 3σ flagging

Aidan in the meeting: *"I'm going to look at the outlier flagging and I'm going to look at maybe some annotations of what happened."*

Two complementary outlier definitions, computed per `(aqsid × parameter_code × year)`:

1. **Tukey extreme outlier** — value > Q3 + 3·IQR or < Q1 − 3·IQR. Robust to skew (criteria pollutants are right-skewed). The `k=3` multiplier filters to *extreme* outliers; `k=1.5` would catch mild ones (commented but not run by default).
2. **3σ rule** — |value − mean| > 3·std. Classic Gaussian outlier; sensitive to skew but useful as a cross-check.

We **only flag the criteria pollutants** (Ozone, NOx_Family, PM2.5, PM10, CO, SO2) at hourly resolution. VOCs and the 24hr-sampled tables follow their own outlier conventions and are out of scope for this notebook.

In [ ]:
outliers = pd.read_sql(text(f"""
    WITH stats AS (
      SELECT aqsid, parameter_code, year,
             AVG(sample_measurement)                                     AS mean,
             STDDEV(sample_measurement)                                   AS std,
             PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY sample_measurement) AS q1,
             PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sample_measurement) AS median,
             PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY sample_measurement) AS q3,
             COUNT(*)                                                     AS n
        FROM aq.pollutant_hourly
       WHERE sample_measurement IS NOT NULL
         AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
       GROUP BY aqsid, parameter_code, year
    )
    SELECT ph.aqsid::text AS aqsid, ph.site_name, ph.county_name,
           ph.pollutant_group, ph.parameter_code, ph.pollutant_name,
           ph.year, ph.date_local, ph.time_local,
           ph.sample_measurement                              AS value,
           s.mean, s.std, s.median, s.q1, s.q3,
           (s.q3 - s.q1)                                      AS iqr,
           (s.q3 + {OUTLIER_K}*(s.q3-s.q1))                   AS upper_tukey,
           (s.q1 - {OUTLIER_K}*(s.q3-s.q1))                   AS lower_tukey,
           CASE WHEN s.std > 0 THEN (ph.sample_measurement - s.mean) / s.std ELSE NULL END AS zscore_year,
           CASE WHEN ph.sample_measurement > s.q3 + {OUTLIER_K}*(s.q3-s.q1)
                  OR ph.sample_measurement < s.q1 - {OUTLIER_K}*(s.q3-s.q1) THEN true ELSE false END AS tukey_extreme,
           CASE WHEN s.std > 0 AND ABS(ph.sample_measurement - s.mean) > 3*s.std THEN true ELSE false END AS sigma3_outlier
      FROM aq.pollutant_hourly ph
      JOIN stats s USING (aqsid, parameter_code, year)
     WHERE ph.sample_measurement IS NOT NULL
       AND ph.year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
       AND ph.pollutant_group IN ('Ozone','NOx_Family','PM2.5','PM10','CO','SO2')
       AND ( ph.sample_measurement > s.q3 + {OUTLIER_K}*(s.q3-s.q1)
          OR ph.sample_measurement < s.q1 - {OUTLIER_K}*(s.q3-s.q1)
          OR (s.std > 0 AND ABS(ph.sample_measurement - s.mean) > 3*s.std) )
"""), engine)
outliers['date_local'] = outliers['date_local'].astype(str)
outliers.to_csv(REPORT_DIR / 'outliers_flagged.csv', index=False)

print(f'Flagged outlier rows: {len(outliers):,}')
print()
print(outliers.groupby(['pollutant_group'])[['tukey_extreme','sigma3_outlier']].sum().to_string())
print()
print('Top 20 by extremity (Tukey distance above the upper fence):')
outliers['tukey_excess'] = (outliers['value'] - outliers['upper_tukey']).clip(lower=0)
display(outliers.sort_values('tukey_excess', ascending=False).head(20)[
    ['aqsid','site_name','pollutant_group','parameter_code','date_local','time_local',
     'value','median','q3','iqr','upper_tukey','tukey_excess','zscore_year']
])

In [ ]:
# Visual: count of Tukey-extreme outliers per pollutant per year
by_pg_yr = (outliers[outliers.tukey_extreme]
    .groupby(['pollutant_group','year']).size().rename('n').reset_index())
pv = by_pg_yr.pivot(index='pollutant_group', columns='year', values='n').fillna(0)
pv.to_csv(REPORT_DIR / 'outlier_counts_pollutant_year.csv')

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(pv, annot=True, fmt='.0f', cmap='YlOrBr',
            linewidths=0.4, linecolor='white',
            cbar_kws={'label':'# Tukey-extreme outlier hours'},
            annot_kws={'fontsize':9}, ax=ax)
ax.set_title(f'Tukey extreme outliers (k={OUTLIER_K}) — pollutant × year',
             color=BRAND_NAVY, fontweight='bold', pad=10)
ax.set_xlabel('year'); ax.set_ylabel('')
fig.tight_layout()
fig.savefig(REPORT_DIR / 'figs' / 'outliers_by_pollutant_year.png', dpi=140, bbox_inches='tight')
plt.show()

## 10. Standardized hourly z-scores — extreme-event detection

Aidan in the meeting: *"a standardized, hourly Z score would be interesting for everything. This way we just see, we can just code the extreme events from that really quickly... like everything that's within 1 to 4 Z score, like 1, 1.5 plus Z scores."*

**Method.** For every `(aqsid × parameter_code × month × hour-of-day)`, compute the historical mean and std across all years 2015–2025. A point's standardized score is `(value − μ) / σ`, where μ and σ come from the matching seasonal-and-circadian bin. This isolates *extreme departures from the diurnal+seasonal norm at the same site*, not just "high values" — a midday June PM2.5 of 30 µg/m³ is normal in Bexar but extreme in November.

We emit the count of hours flagged at each threshold per (site × parameter).

In [ ]:
# Step 1: compute (aqsid × parameter_code × month × hour) baseline (μ, σ)
# then emit a count of values above each |z| threshold per site × parameter.
# Server-side, single query, scales to all criteria pollutants.
z_counts = pd.read_sql(text(f"""
    WITH bl AS (
      SELECT aqsid, parameter_code, month, hour,
             AVG(sample_measurement)    AS mean,
             STDDEV(sample_measurement) AS std
        FROM aq.pollutant_hourly
       WHERE sample_measurement IS NOT NULL
         AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
         AND pollutant_group IN ('Ozone','NOx_Family','PM2.5','PM10','CO','SO2')
       GROUP BY aqsid, parameter_code, month, hour
      HAVING COUNT(*) >= 30 AND STDDEV(sample_measurement) > 0
    ),
    flagged AS (
      SELECT ph.aqsid::text AS aqsid, ph.site_name, ph.pollutant_group,
             ph.parameter_code, ph.pollutant_name,
             ABS((ph.sample_measurement - bl.mean) / bl.std) AS abs_z
        FROM aq.pollutant_hourly ph
        JOIN bl USING (aqsid, parameter_code, month, hour)
       WHERE ph.sample_measurement IS NOT NULL
         AND ph.year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    )
    SELECT aqsid, site_name, pollutant_group, parameter_code, pollutant_name,
           COUNT(*) AS n_observations,
           SUM(CASE WHEN abs_z >= 1.5 THEN 1 ELSE 0 END) AS n_abs_z_ge_1p5,
           SUM(CASE WHEN abs_z >= 2.0 THEN 1 ELSE 0 END) AS n_abs_z_ge_2p0,
           SUM(CASE WHEN abs_z >= 3.0 THEN 1 ELSE 0 END) AS n_abs_z_ge_3p0,
           SUM(CASE WHEN abs_z >= 4.0 THEN 1 ELSE 0 END) AS n_abs_z_ge_4p0,
           MAX(abs_z) AS max_abs_z
      FROM flagged
      GROUP BY aqsid, site_name, pollutant_group, parameter_code, pollutant_name
      ORDER BY n_abs_z_ge_3p0 DESC
"""), engine)
z_counts.to_csv(REPORT_DIR / 'zscore_counts_per_site_param.csv', index=False)
print(f'Site × parameter combinations with computed z-baseline: {len(z_counts):,}')
print()
print('Headline thresholds (sum across all sites × parameters):')
print(f'  |z| >= 1.5 : {z_counts.n_abs_z_ge_1p5.sum():>10,}')
print(f'  |z| >= 2.0 : {z_counts.n_abs_z_ge_2p0.sum():>10,}')
print(f'  |z| >= 3.0 : {z_counts.n_abs_z_ge_3p0.sum():>10,}')
print(f'  |z| >= 4.0 : {z_counts.n_abs_z_ge_4p0.sum():>10,}')
print()
print('Top 15 site × parameter combos by # of |z| >= 3 hours:')
display(z_counts.head(15))

In [ ]:
# Pull the top 200 most-extreme z-scored hours across the network for manual review.
extremes = pd.read_sql(text(f"""
    WITH bl AS (
      SELECT aqsid, parameter_code, month, hour,
             AVG(sample_measurement)    AS mean,
             STDDEV(sample_measurement) AS std
        FROM aq.pollutant_hourly
       WHERE sample_measurement IS NOT NULL
         AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
         AND pollutant_group IN ('Ozone','NOx_Family','PM2.5','PM10','CO','SO2')
       GROUP BY aqsid, parameter_code, month, hour
      HAVING COUNT(*) >= 30 AND STDDEV(sample_measurement) > 0
    )
    SELECT ph.aqsid::text AS aqsid, ph.site_name, ph.county_name,
           ph.pollutant_group, ph.parameter_code, ph.pollutant_name,
           ph.date_local, ph.time_local, ph.sample_measurement AS value,
           bl.mean, bl.std,
           (ph.sample_measurement - bl.mean) / bl.std AS zscore
      FROM aq.pollutant_hourly ph
      JOIN bl USING (aqsid, parameter_code, month, hour)
     WHERE ph.sample_measurement IS NOT NULL
       AND ph.year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
       AND ABS((ph.sample_measurement - bl.mean) / bl.std) >= 4.0
     ORDER BY ABS((ph.sample_measurement - bl.mean) / bl.std) DESC
     LIMIT 200
"""), engine)
extremes['date_local'] = extremes['date_local'].astype(str)
extremes.to_csv(REPORT_DIR / 'extreme_events_top200.csv', index=False)
print(f'Top {len(extremes)} extreme events (|z| ≥ 4):')
display(extremes.head(15))

## 11. Descriptive statistics — per pollutant × site × year

Standard descriptives per `(aqsid × parameter_code × year)`: n, mean, std, median, IQR, p5/p25/p75/p95/p99, max, plus the corresponding NAAQS threshold (where defined). Use this as the lookup table for the manuscript's Table 1 / Supplementary tables.

NAAQS thresholds (40 CFR Part 50):

| Pollutant | NAAQS level | Form |
|---|---|---|
| Ozone | 0.070 ppm | 4th-highest daily 8-hr avg (3-yr) |
| NO₂ | 0.100 ppm | 1-hr p98 (3-yr) |
| SO₂ | 0.075 ppm | 1-hr p99 (3-yr) |
| CO | 9 ppm | 8-hr; not exceeded > 1×/yr |
| PM₂.₅ | 9 µg/m³ | annual mean |
| PM₁₀ | 150 µg/m³ | 24-hr; not exceeded > 1×/yr |

In [ ]:
descr = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, site_name, county_name, pollutant_group,
           parameter_code, pollutant_name, year::int AS year,
           COUNT(*)                                                     AS n,
           AVG(sample_measurement)                                      AS mean,
           STDDEV(sample_measurement)                                   AS std,
           MIN(sample_measurement)                                      AS min,
           MAX(sample_measurement)                                      AS max,
           PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY sample_measurement) AS p5,
           PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY sample_measurement) AS q1,
           PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sample_measurement) AS median,
           PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY sample_measurement) AS q3,
           PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY sample_measurement) AS p95,
           PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY sample_measurement) AS p99
    FROM aq.pollutant_hourly
    WHERE sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group,
             parameter_code, pollutant_name, year
    ORDER BY pollutant_group, parameter_code, year, aqsid
"""), engine)
descr['iqr'] = descr['q3'] - descr['q1']
descr.to_csv(REPORT_DIR / 'descriptive_stats.csv', index=False)
print(f'Descriptive rows: {len(descr):,}')
print()
print('=== Per-pollutant network-wide summary (all sites, all years) ===')
net = (descr.groupby(['pollutant_group','parameter_code','pollutant_name'])
    .agg(n=('n','sum'),
         sites=('aqsid','nunique'),
         mean=('mean', lambda s: round(np.average(s, weights=descr.loc[s.index, 'n']), 4)),
         median=('median','mean'),
         p95=('p95','mean'),
         p99=('p99','mean'),
         max=('max','max')))
print(net.to_string())
descr.head()

## 12. Loss recovery plan & action items

Concrete next steps Aidan can execute this week. Each block links a finding to a `pipeline/` change.

In [ ]:
from IPython.display import Markdown, display

no_raw = gaps[gaps.recovery_status=='NO_RAW_DATA_FOUND']
n_outlier_total = len(outliers)
n_outlier_tukey = int(outliers.tukey_extreme.sum())
n_outlier_sigma = int(outliers.sigma3_outlier.sum())
n_z3 = int(z_counts.n_abs_z_ge_3p0.sum())

# Files that would actually add rows when ingested
high_impact_files = by_file_impact[by_file_impact.delta_rows > 1000].index.tolist()
total_net_new = int(refresh.delta.clip(lower=0).sum())

md = f"""
### A. Ingest the high-impact raw refresh files
The §6b refresh-impact analysis showed **{total_net_new:,} net new rows** could be added to v0.4.0 just by running `step_01b_ingest_tceq_raw.py` against the raw files. The high-impact files (delta > 1,000 rows):

{chr(10).join(f'- `{f}` — adds {int(by_file_impact.loc[f,"delta_rows"]):,} rows ({int(by_file_impact.loc[f,"n_sites"])} sites)' for f in high_impact_files)}

The other 7 files show delta = 0 (already ingested) or negative delta (small 2025 Q4 deltas already superseded by the full-year files). Skip them.

### B. EPA-only legacy data — {len(no_raw)} cells, {int(no_raw.v037_epa_rows.sum()):,} rows
The genuinely lost EPA-only data lives at sites that **do not report hourly to TCEQ TAMIS** (Camp Bullis, San Antonio Northwest, Converse, Brownsville, Mission, Edinburg, Eagle Pass, Karnes City, Kingsville, Pleasanton, Laredo, Floresville, etc.). Two paths:

1. **Keep them in `aq_v0_3_7_epa.*`** (already done) and document in `pipeline/docs/06_data_quality.md`.
2. **Build the dual-source v0.4.1 schema** Aidan proposed in the meeting: *"I might just port in the old EPA data for those sites and keep the new one and then just add a source column."* That's the next pipeline iteration; this notebook surfaces *which* cells need that path.

### C. Fix the cadence routing in `step_01b`
{len(cadence)} (aqsid × parameter × year) cells in v0.4.0 have <200 rows in `pollutant_hourly`, suggesting they're 24hr-sampled but landed in the wrong table. Update `step_01b_ingest_tceq_raw.py` to route Sample-Duration-Code-7 rows into `pollutant_daily_24hr` (per decision #19) **regardless of which pollutant_group they belong to**, not just PM10 at site 0060.

### D. The >100% completeness fix
§7 corrects the metric — divide by parameter_code, not by pollutant_group. Update the v0.4.0 audit notebook + the public docs to use per-parameter completeness. The previously \"shocking ~300% NOx coverage\" disappears once you separate NO, NO₂, NOx.

### E. Outlier review queue
Outlier flagging produced **{n_outlier_total:,} rows** ({n_outlier_tukey:,} Tukey-extreme, {n_outlier_sigma:,} 3-σ). The `extreme_events_top200.csv` is the manual-review list. Pair with the weather hourly join (the `AQ_Weather_SiteMapping.csv` lookup) to annotate which outliers coincided with documented weather events.

### F. Extreme-event z-score counts
§10's `zscore_counts_per_site_param.csv` is your ready-to-go input for the *\"flag the extreme events at |z| ≥ 1.5 / 2.0 / 3.0\"* analysis Aidan described. Pivot it by `site × month` to surface seasonal anomalies.

---

### Standing to-dos (not closed here — owners listed)

| # | Item | Owner |
|---|---|---|
| A | TCEQ AQS web parameter survey for Jan 1 2024 baseline | Jasleen |
| B | Predictive-modeling sites + train/validate split | Manasa (loop in Dr. Jin) |
| C | Dual-source `pollutant_hourly_epa` table for NAAQS calcs | Aidan (next sprint) |
| D | Weather-correlation table for the outlier review list | Aidan or Manasa |
| E | Manuscript methodology blurb on OpenWeather data sourcing | Aidan |
| F | Separate TCEQ pull for the EPA-only sites that are still in TAMIS | Aidan |
"""
display(Markdown(md))

## 13. Export everything to a single shareable HTML report

In [ ]:
import subprocess
candidates = [Path.cwd() / 'AM_Lab_Meeting_Followup.ipynb']
candidates += list(Path('/content').rglob('AM_Lab_Meeting_Followup.ipynb')) if Path('/content').exists() else []
nb_path = next((p for p in candidates if p.exists()), None)
html_out = REPORT_DIR / 'AM_Lab_Meeting_Followup.html'
if nb_path:
    res = subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'html',
                          '--output', str(html_out), str(nb_path)],
                         capture_output=True, text=True)
    if res.returncode == 0:
        print(f'OK HTML report -> {html_out}')
    else:
        print('nbconvert stderr:', res.stderr[:500])
else:
    print('Notebook path not found; skip HTML export.')

print()
print('=== All outputs in', REPORT_DIR, '===')
for p in sorted(REPORT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(REPORT_DIR)}  ({p.stat().st_size/1024:.1f} kB)')

---

**End of follow-up.** Bring the HTML + `epa_gap_recovery_plan.csv` to the next meeting. Open standing to-dos roll into next week's plan.

**Companion notebooks:**
- `AM_Week1_Report.ipynb` — Week 1 baseline diurnal + descriptives + geospatial
- `AM_Data_Availability_Audit.ipynb` — v0.4.0 completeness + Gantt dashboard
- `AM_v037_vs_v040_Audit.ipynb` — v0.3.7 ↔ v0.4.0 diff (this notebook builds on it)